# MLOps and Delivery: Package, Monitor and Decide

Notebook for **Chapter 8** of the *ways* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ways/ch08.html

Tip: change a number, predict the output, then run the cell again.

In [ ]:
# Setup: the files used in this chapter are downloaded next to the notebook.
import os, urllib.request
os.makedirs("data", exist_ok=True)
for f in ["customers.csv", "orders.csv", "products.csv", "shopkart.db"]:
    if not os.path.exists("data/" + f):
        urllib.request.urlretrieve("https://dadipavan.github.io/Learning/study/ways/data/" + f, "data/" + f)


## 8.1 From notebook to product

A model in a notebook is an experiment. A model in production is a small system with five parts, each with an owner.

## 8.2 Package: save the pipeline and its record

The unit you save must include *preprocessing*. Saving only the model, and re-creating the encoders by hand in the serving code, is the most common cause of "works in the notebook, wrong in production" (training-serving skew). With scikit-learn the Pipeline object bundles both. Save it with `joblib` and write a **metadata** file next to it: training window, features, metrics, library versions and a hash of the training data.

In [ ]:
DATA = "data/"
import sys, os, json, hashlib, tempfile
sys.path.insert(0, DATA)
import prep, mlprep
import numpy as np, sklearn, joblib
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

o, c, p = prep.load_clean(DATA)
d = mlprep.late_frame(o)
tr, te = d[d.order_date < "2024-10-01"], d[d.order_date >= "2024-10-01"]
F = mlprep.CAT + mlprep.NUM
pre = ColumnTransformer([("c", OneHotEncoder(handle_unknown="ignore"), mlprep.CAT), ("n", StandardScaler(), mlprep.NUM)])
model = make_pipeline(pre, LogisticRegression(max_iter=1000)).fit(tr[F], tr.late)

folder = tempfile.mkdtemp()
joblib.dump(model, os.path.join(folder, "late_model.joblib"))
meta = {"name": "late_delivery", "version": "1.0", "train_window": ["2024-01-01", "2024-09-30"], "features": F,
        "test_auc": round(roc_auc_score(te.late, model.predict_proba(te[F])[:, 1]), 4), "threshold": 0.11,
        "sklearn": sklearn.__version__, "rows": len(tr),
        "data_hash": hashlib.sha256(tr[F].to_csv().encode()).hexdigest()[:12]}
json.dump(meta, open(os.path.join(folder, "meta.json"), "w"), indent=1)

again = joblib.load(os.path.join(folder, "late_model.joblib"))
same = np.array_equal(model.predict_proba(te[F]), again.predict_proba(te[F]))
print("files:", sorted(os.listdir(folder)))
print("identical predictions after reload:", same)
print({k: meta[k] for k in ("version", "train_window", "test_auc", "threshold", "rows")})

files: ['late_model.joblib', 'meta.json']
identical predictions after reload: True
{'version': '1.0', 'train_window': ['2024-01-01', '2024-09-30'], 'test_auc': 0.8438, 'threshold': 0.11, 'rows': 12966}


## 8.3 Serve: a validated prediction function

A prediction function is the simplest serving interface: input dictionary in, decision out. Production adds three things the notebook never had: **input validation**, **explicit handling of unknowns** and **tests**.

In [ ]:
DATA = "data/"
import sys, os, json, tempfile, time
sys.path.insert(0, DATA)
import prep, mlprep
import pandas as pd, joblib
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline

o, c, p = prep.load_clean(DATA)
d = mlprep.late_frame(o)
tr = d[d.order_date < "2024-10-01"]
F = mlprep.CAT + mlprep.NUM
pre = ColumnTransformer([("c", OneHotEncoder(handle_unknown="ignore"), mlprep.CAT), ("n", StandardScaler(), mlprep.NUM)])
MODEL = make_pipeline(pre, LogisticRegression(max_iter=1000)).fit(tr[F], tr.late)
THRESHOLD, CITIES = 0.11, {"Bengaluru", "Mumbai", "Delhi", "Chennai", "Pune"}

def predict_late(order):
    """order: dict with city, channel, category, quantity, discount, order_hour, promised_minutes, weekend."""
    errors = []
    if order.get("city") not in CITIES: errors.append("unknown city")
    if order.get("channel") not in {"App", "Web", "Unknown"}: errors.append("bad channel")
    if not (isinstance(order.get("quantity"), (int, float)) and 1 <= order["quantity"] <= 20): errors.append("quantity out of range")
    if order.get("promised_minutes") not in (45, 60): errors.append("promise must be 45 or 60")
    if not (0 <= order.get("order_hour", -1) <= 23): errors.append("hour out of range")
    if errors:
        return {"ok": False, "errors": errors}
    row = dict(order, evening=int(18 <= order["order_hour"] <= 21))
    prob = float(MODEL.predict_proba(pd.DataFrame([row])[F])[:, 1][0])
    return {"ok": True, "p_late": round(prob, 3), "action": "intervene" if prob >= THRESHOLD else "none"}

good = dict(city="Bengaluru", channel="App", category="Grocery", quantity=2, discount=0.1, order_hour=19, promised_minutes=45, weekend=0)
print(predict_late(good))
print(predict_late(dict(good, order_hour=11, promised_minutes=60, city="Chennai")))
print(predict_late(dict(good, city="Atlantis", quantity=-3)))

# tests: the kind you keep in a repository
assert predict_late(good)["action"] == "intervene"
assert predict_late(dict(good, promised_minutes=60, order_hour=11))["action"] == "none"
assert not predict_late(dict(good, quantity=500))["ok"]
t = time.perf_counter()
for _ in range(200):
    predict_late(good)
ms = (time.perf_counter() - t) / 200 * 1000
print("single-order latency under 50 ms:", ms < 50)

{'ok': True, 'p_late': 0.548, 'action': 'intervene'}
{'ok': True, 'p_late': 0.006, 'action': 'none'}
{'ok': False, 'errors': ['unknown city', 'quantity out of range']}
single-order latency under 50 ms: True


## 8.4 Monitor: has the world changed?

A model is a snapshot of the past. Two things go wrong after launch. **Data drift**: the inputs change (more orders in the evening, a new city). **Concept drift**: the relationship between inputs and the outcome changes (a new rider contract changes delivery times). Drift in inputs can be seen immediately, because inputs need no labels; performance can only be measured once outcomes are known.

In [ ]:
DATA = "data/"
import sys
sys.path.insert(0, DATA)
import prep, mlprep
import numpy as np, pandas as pd

def psi(ref, new, eps=1e-4):
    ref, new = np.asarray(ref, float), np.asarray(new, float)
    a, b = np.clip(ref / ref.sum(), eps, None), np.clip(new / new.sum(), eps, None)
    return float(np.sum((b - a) * np.log(b / a)))
print("hand example:", round(psi([50, 30, 20], [40, 30, 30]), 5))

o, c, p = prep.load_clean(DATA)
d = mlprep.late_frame(o)
ref, new = d[d.order_date < "2024-07-01"], d[d.order_date >= "2024-10-01"]

def psi_cat(col):
    cats = sorted(d[col].dropna().unique())
    return psi([(ref[col] == k).sum() for k in cats], [(new[col] == k).sum() for k in cats])
def psi_num(col, bins=10):
    edges = np.unique(np.quantile(ref[col].dropna(), np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    return psi(np.histogram(ref[col].dropna(), edges)[0], np.histogram(new[col].dropna(), edges)[0])

rows = [(c_, "categorical", psi_cat(c_)) for c_ in ["city", "channel", "category", "promised_minutes"]]
rows += [(c_, "numeric", psi_num(c_)) for c_ in ["order_hour", "quantity", "discount"]]
t = pd.DataFrame(rows, columns=["feature", "type", "psi"])
t["status"] = np.where(t.psi < 0.1, "stable", np.where(t.psi < 0.25, "watch", "shift"))
print(t.round(4).to_string(index=False))

hand example: 0.06286
         feature        type    psi status
            city categorical 0.0012 stable
         channel categorical 0.0001 stable
        category categorical 0.0029 stable
promised_minutes categorical 0.0001 stable
      order_hour     numeric 0.0017 stable
        quantity     numeric 0.0001 stable
        discount     numeric 0.0006 stable


In [ ]:
DATA = "data/"
import sys
sys.path.insert(0, DATA)
import prep, mlprep
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

o, c, p = prep.load_clean(DATA)
d = mlprep.late_frame(o)
F = mlprep.CAT + mlprep.NUM
def fit(df):
    pre = ColumnTransformer([("c", OneHotEncoder(handle_unknown="ignore"), mlprep.CAT), ("n", StandardScaler(), mlprep.NUM)])
    return make_pipeline(pre, LogisticRegression(max_iter=1000)).fit(df[F], df.late)

frozen = fit(d[d.order_date < "2024-07-01"])
rows = []
for m in range(7, 13):
    month = d[d.order_date.dt.month == m]
    retrained = fit(d[d.order_date < f"2024-{m:02d}-01"])
    rows.append((f"2024-{m:02d}", len(month), month.late.mean(),
                 roc_auc_score(month.late, frozen.predict_proba(month[F])[:, 1]),
                 roc_auc_score(month.late, retrained.predict_proba(month[F])[:, 1])))
t = pd.DataFrame(rows, columns=["month", "orders", "late_rate", "auc_frozen", "auc_retrained_monthly"])
print(t.round(4).to_string(index=False))
print("mean AUC frozen:", round(t.auc_frozen.mean(), 4), "| retrained monthly:", round(t.auc_retrained_monthly.mean(), 4))

month  orders  late_rate  auc_frozen  auc_retrained_monthly
2024-07    1732     0.1386      0.8155                 0.8155
2024-08    1641     0.1292      0.8407                 0.8414
2024-09    1423     0.1300      0.8558                 0.8547
2024-10    1332     0.1366      0.8468                 0.8481
2024-11    1615     0.1511      0.8399                 0.8408
2024-12    1295     0.1359      0.8445                 0.8437
mean AUC frozen: 0.8405 | retrained monthly: 0.8407


## 8.5 Retrain: triggers and rules

Retraining on a calendar is easy but may waste effort or arrive too late. Triggers based on evidence are better.

## 8.6 Document: the model card and the decision memo

Two documents travel with a model. The **model card** is for colleagues and auditors: what the model is for, how it was built, how it performs, where it should not be used. The **decision memo** is for the person who asked the question in the first place. Both can be generated from the same numbers the analysis produced, which prevents the commonest memo error: a number typed by hand that no longer matches the data.

In [ ]:
DATA = "data/"
import sys
sys.path.insert(0, DATA)
import prep, mlprep
import pandas as pd

o, c, p = prep.load_clean(DATA)
rev = o.revenue.sum()
cat = o.groupby("category").revenue.sum().sort_values(ascending=False)
city = o.groupby("city").revenue.sum().sort_values(ascending=False)
m = o.groupby(o.order_date.dt.month).revenue.sum()
ret = o.groupby("category").apply(lambda d: (d.status == "returned").mean(), include_groups=False).sort_values(ascending=False)
late = o.late.mean()
per = o.groupby("customer_id").size()
memo = f"""MEMO to the CEO: ShopKart 2024 in five lines
1. Size: {len(o):,} orders, rupees {rev/1e6:.1f} million net revenue (delivered orders, after cleaning).
2. Where: {cat.index[0]} ({cat.iloc[0]/rev:.0%}) and {cat.index[1]} ({cat.iloc[1]/rev:.0%}) lead; {city.index[0]} is the top city ({city.iloc[0]/rev:.0%}).
3. Trend: revenue peaked in month {m.idxmax()} and fell in the autumn; a late-November promotion lifted November by {m[11]/m[10]-1:.0%} over October.
4. Service: {late:.1%} of deliveries miss their promise; 45-minute promises are late {o[o.promised_minutes==45].late.mean():.0%} of the time against {o[o.promised_minutes==60].late.mean():.0%} for 60-minute ones.
5. Returns: {(o.status=='returned').mean():.1%} overall; {ret.index[0]} is highest at {ret.iloc[0]:.0%}.
Customers: {(per>=2).mean():.0%} of customers who ordered placed at least two orders; {(~c.customer_id.isin(o.customer_id)).sum():,} registered customers never ordered.
RECOMMENDATIONS
 a. Review the 45-minute promise: it accounts for most late deliveries; test 60 minutes in one city with a randomised experiment (about 19,000 orders per arm needed to see a one-point change).
 b. Use the late-delivery model (AUC about 0.84) to trigger rescue actions at a cost-based threshold; re-check recall by city before rollout.
 c. Reduce Fashion returns before spending on acquisition there.
 d. Win back the never-ordered customers with a first-order offer, measured with a holdout group.
LIMITS: one year of data, five cities, no cost of goods per delivery, discounts not shown to raise basket size."""
print(memo)

MEMO to the CEO: ShopKart 2024 in five lines
1. Size: 19,995 orders, rupees 17.3 million net revenue (delivered orders, after cleaning).
2. Where: Fashion (30%) and Electronics (28%) lead; Bengaluru is the top city (28%).
3. Trend: revenue peaked in month 8 and fell in the autumn; a late-November promotion lifted November by 15% over October.
4. Service: 13.8% of deliveries miss their promise; 45-minute promises are late 25% of the time against 3% for 60-minute ones.
5. Returns: 14.8% overall; Fashion is highest at 23%.
Customers: 81% of customers who ordered placed at least two orders; 785 registered customers never ordered.
RECOMMENDATIONS
 a. Review the 45-minute promise: it accounts for most late deliveries; test 60 minutes in one city with a randomised experiment (about 19,000 orders per arm needed to see a one-point change).
 b. Use the late-delivery model (AUC about 0.84) to trigger rescue actions at a cost-based threshold; re-check recall by city before rollout.
 c. Reduce Fash

## 8.7 Choose your path by role

The journey of this document is the shared skeleton. Roles differ in where they spend their time. Pick the path that matches the job you want and read the other study documents on this site for depth (listed as "Next reading").

## 8.8 The one-page end-to-end checklist

## Chapter recap

## Practice set

Problem 8.9.1 Why must the preprocessing be saved together with the model?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ways/ch08.html